# Supervised Classification

Trains a classification model on tabular survey columns. Handles automatic preprocessing, SHAP feature importance, cross-validation, and narrative output. Saves predicted class labels back to SuAVE as a new column.

**Models (scikit-learn):** Logistic Regression, Random Forest, Gradient Boosting, XGBoost, AdaBoost, Extra Trees, SVM, K-Nearest Neighbors, Decision Tree.  
**Statistical model:** Logit with p-values, confidence intervals, pseudo-R2 (statsmodels).

Run the five parameter cells first, then work through Steps 1-5.

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# -- Colab only: mount Google Drive to load session credentials ------------
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# -- Load SuAVE parameters -------------------------------------------------
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# -- Variables used throughout this notebook --------------------------------
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin  = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)
su.show_params(_p)


In [ ]:
import sys, subprocess
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint
from collections import OrderedDict as _OD

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     KFold, cross_val_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    r2_score, mean_squared_error, mean_absolute_error)
from sklearn.linear_model import (LogisticRegression, Ridge, Lasso,
                                   ElasticNet, LinearRegression)
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    ExtraTreesClassifier, ExtraTreesRegressor)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import statsmodels.api as sm

for _pkg in ('xgboost', 'shap'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'Installing {_pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])
from xgboost import XGBClassifier, XGBRegressor
import shap

printmd('**Imports complete.**')


In [ ]:
df_raw = panellibs.extract_data(absolutePath + csv_file)
print(f"Survey: {csv_file}  |  {len(df_raw):,} rows  |  {len(df_raw.columns)} columns")

_SKIP = ('#img', '#link', '#href', '#netvis', '#hidden', '#hiddenmore',
         '#name', '#lat', '#lon', '#coord', '#textlocation')

def _usable(c):   return not any(q in c for q in _SKIP)
def _isnumcol(c): return '#number' in c
def _lbl(c):      return c.split('#')[0].strip() or c

usable   = [c for c in df_raw.columns if _usable(c)]
num_raw  = [c for c in usable if _isnumcol(c)]
cat_raw  = [c for c in usable if not _isnumcol(c)]
_lbl2raw = {_lbl(c): c for c in usable}
_labels  = list(_lbl2raw.keys())

rows = []
for c in usable:
    rows.append({'Column': _lbl(c),
                 'Type': 'numeric' if _isnumcol(c) else 'categorical',
                 'Unique values': df_raw[c].nunique(),
                 'Missing': df_raw[c].isnull().sum()})
display(pd.DataFrame(rows)
        .style.set_properties(**{'font-size': '11px'})
        .set_caption(f'{len(usable)} usable columns  '
                     f'({len(num_raw)} numeric, {len(cat_raw)} categorical)'))


In [ ]:
# -- Step 1: select target, features, and split ----------------------------
y_dd = widgets.Dropdown(
    options=_labels, description='Target y:',
    style={'description_width': '100px'}, layout=widgets.Layout(width='380px'))
X_ms = widgets.SelectMultiple(
    options=_labels, value=_labels[1:min(7, len(_labels))],
    description='Features X:', rows=min(12, len(_labels)),
    style={'description_width': '100px'},
    layout=widgets.Layout(width='380px', height='210px'))
split_s = widgets.FloatSlider(
    value=0.2, min=0.05, max=0.5, step=0.05, description='Test set:',
    readout_format='.0%', style={'description_width': '100px'},
    layout=widgets.Layout(width='380px'))
seed_t = widgets.IntText(
    value=42, description='Seed:',
    style={'description_width': '100px'}, layout=widgets.Layout(width='220px'))
_y_info = widgets.HTML()

def _update_y_info(change):
    col = _lbl2raw[y_dd.value]
    n   = df_raw[col].nunique()
    classes = sorted(df_raw[col].dropna().unique())[:8]
    shown = ', '.join(str(c) for c in classes) + ('...' if n > 8 else '')
    if n > 20:
        color, note = '#e07000', f'{n} unique values -- consider regression instead.'
    elif n == 2:
        color, note = '#10b981', 'Binary target -- ROC AUC will be reported.'
    else:
        color, note = '#0ea5e9', f'{n}-class target.'
    _y_info.value = (
        f'<div style="font-size:12px;color:{color};padding:3px 0">'
        f'{note}<br><small style="color:#6b7280">Classes: {shown}</small></div>')

y_dd.observe(_update_y_info, names='value')
_update_y_info(None)

printmd('**Step 1 -- Choose target variable, features, and train/test split:**')
display(widgets.VBox([y_dd, _y_info, X_ms, split_s, seed_t]))


In [ ]:
_hp_widgets = {}

def _make_widget(pname, spec):
    W = widgets.Layout(width='440px')
    S = {'description_width': '160px'}
    kind = spec[0]
    if kind == 'flog':
        _, lo, hi, default, _desc = spec
        default = max(default, lo)
        return widgets.FloatLogSlider(
            value=default, base=10, min=np.log10(lo), max=np.log10(hi),
            step=0.05, description=pname + ':', readout_format='.3g',
            style=S, layout=W)
    elif kind == 'float':
        _, lo, hi, step, default, _desc = spec
        return widgets.FloatSlider(
            value=default, min=lo, max=hi, step=step,
            description=pname + ':', readout_format='.3f', style=S, layout=W)
    elif kind == 'int':
        _, lo, hi, default, _desc = spec
        return widgets.IntSlider(
            value=default, min=lo, max=hi,
            description=pname + ':', style=S, layout=W)
    elif kind == 'drop':
        _, opts, default, _desc = spec
        return widgets.Dropdown(
            options=[str(o) for o in opts], value=str(default),
            description=pname + ':', style=S,
            layout=widgets.Layout(width='440px'))
    return widgets.Label(pname)

_model_info = widgets.HTML()
_params_box  = widgets.VBox([])

def _on_model_change(change):
    name = model_dd.value
    _hp_widgets.clear()
    if name not in _CATALOG:
        _model_info.value = _SM_BLURBS.get(name, '')
        _params_box.children = []
        return
    spec   = _CATALOG[name]
    _url   = spec['url']
    _blurb = spec['blurb']
    _model_info.value = (
        '<div style="font-size:12px;padding:4px 0">'
        f'<a href="{_url}" target="_blank">{name} documentation</a> -- {_blurb}'
        '</div>')
    rows = []
    for pname, pspec in spec['params'].items():
        w = _make_widget(pname, pspec)
        _hp_widgets[pname] = w
        rows.append(w)
    _params_box.children = rows

def _hp_vals():
    out = {}
    for k, w in _hp_widgets.items():
        v = w.value
        if isinstance(v, str):
            if v == 'None':
                v = None
            else:
                try:    v = int(v)
                except ValueError:
                    try: v = float(v)
                    except ValueError: pass
        out[k] = v
    return out

# -- Step 2: select model and configure hyperparameters --------------------
_CATALOG = _OD([
  ('Logistic Regression', dict(
    cls=LogisticRegression,
    fit_kw=dict(solver='saga', random_state=42, n_jobs=-1),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html',
    blurb='Linear model for class probabilities. Fast and interpretable; l1 penalty gives automatic feature selection.',
    shap='linear',
    params=_OD([
      ('C',        ('flog', 1e-3, 1e2, 1.0,  'Inverse regularization (smaller = stronger)')),
      ('penalty',  ('drop', ['l2', 'l1', 'none'], 'l2', 'Regularization type')),
      ('max_iter', ('int',  100, 5000, 1000,  'Solver iteration limit')),
    ]))),
  ('Random Forest', dict(
    cls=RandomForestClassifier,
    fit_kw=dict(random_state=42, n_jobs=-1),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html',
    blurb='Ensemble of decision trees with random feature subsets. Robust to irrelevant features and outliers.',
    shap='tree',
    params=_OD([
      ('n_estimators',      ('int',  50, 500, 100, 'Number of trees')),
      ('max_depth',         ('drop', [2,3,5,7,10,15,20,'None'], 5, 'Max tree depth (None=full)')),
      ('min_samples_split', ('int',   2,  20,   2, 'Min samples to split a node')),
      ('max_features',      ('drop', ['sqrt','log2','None'], 'sqrt', 'Features per split')),
    ]))),
  ('Gradient Boosting', dict(
    cls=GradientBoostingClassifier,
    fit_kw=dict(random_state=42),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html',
    blurb='Sequential tree boosting -- each tree corrects the residuals of the previous. Powerful with careful tuning.',
    shap='tree',
    params=_OD([
      ('n_estimators',  ('int',   50, 500, 100, 'Number of boosting stages')),
      ('learning_rate', ('float', 0.01, 0.5, 0.05, 0.1, 'Shrinkage per tree')),
      ('max_depth',     ('drop',  [2,3,4,5,6,8], 3, 'Max depth per tree')),
      ('subsample',     ('float', 0.5, 1.0, 0.1, 1.0, 'Row fraction per tree (<1 = stochastic)')),
    ]))),
  ('XGBoost', dict(
    cls=XGBClassifier,
    fit_kw=dict(random_state=42, eval_metric='logloss', verbosity=0),
    url='https://xgboost.readthedocs.io/en/stable/parameter.html',
    blurb='Optimized gradient boosting with L1/L2 regularization and native missing-value handling.',
    shap='tree',
    params=_OD([
      ('n_estimators',  ('int',   50, 500, 100, 'Number of trees')),
      ('learning_rate', ('float', 0.01, 0.5, 0.05, 0.1, 'Step size shrinkage')),
      ('max_depth',     ('int',   2, 12, 6,  'Max depth per tree')),
      ('reg_alpha',     ('float', 0.0, 5.0, 0.5, 0.0, 'L1 weight regularization')),
      ('reg_lambda',    ('float', 0.0, 5.0, 0.5, 1.0, 'L2 weight regularization')),
    ]))),
  ('AdaBoost', dict(
    cls=AdaBoostClassifier,
    fit_kw=dict(random_state=42, algorithm='SAMME'),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html',
    blurb='Re-weights misclassified samples so each subsequent weak learner focuses on the hardest examples.',
    shap='tree',
    params=_OD([
      ('n_estimators',  ('int',   10, 300, 50, 'Number of weak learners')),
      ('learning_rate', ('float', 0.01, 2.0, 0.1, 1.0, 'Contribution weight per learner')),
    ]))),
  ('Extra Trees', dict(
    cls=ExtraTreesClassifier,
    fit_kw=dict(random_state=42, n_jobs=-1),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html',
    blurb='Like Random Forest but splits are chosen fully at random -- faster and sometimes more accurate on high-dimensional data.',
    shap='tree',
    params=_OD([
      ('n_estimators',      ('int',  50, 500, 100, 'Number of trees')),
      ('max_depth',         ('drop', [2,3,5,7,10,15,20,'None'], 'None', 'Max tree depth')),
      ('min_samples_split', ('int',   2,  20,   2, 'Min samples to split a node')),
    ]))),
  ('SVM', dict(
    cls=SVC,
    fit_kw=dict(probability=True, random_state=42),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html',
    blurb='Maximum-margin classifier. Effective in high-dimensional spaces; slow on datasets above ~10k rows.',
    shap='kernel',
    params=_OD([
      ('C',      ('flog', 1e-2, 1e3, 1.0, 'Regularization (larger = harder margin)')),
      ('kernel', ('drop', ['rbf','linear','poly'], 'rbf', 'Kernel function')),
      ('gamma',  ('drop', ['scale','auto'], 'scale', 'Kernel coefficient')),
    ]))),
  ('K-Nearest Neighbors', dict(
    cls=KNeighborsClassifier,
    fit_kw=dict(n_jobs=-1),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html',
    blurb='Classifies by majority vote among k nearest training examples. No training phase; sensitive to feature scale.',
    shap='kernel',
    params=_OD([
      ('n_neighbors', ('int',  1, 30, 5, 'Number of neighbors')),
      ('weights',     ('drop', ['uniform','distance'], 'uniform', 'Neighbor weighting')),
      ('metric',      ('drop', ['euclidean','manhattan','minkowski'], 'euclidean', 'Distance metric')),
    ]))),
  ('Decision Tree', dict(
    cls=DecisionTreeClassifier,
    fit_kw=dict(random_state=42),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html',
    blurb='Single interpretable tree. Prone to overfitting without depth limits; useful as a transparent baseline.',
    shap='tree',
    params=_OD([
      ('max_depth',         ('drop', [2,3,4,5,6,8,10,'None'], 5, 'Max depth (None=full)')),
      ('min_samples_split', ('int',   2, 20, 2, 'Min samples to split a node')),
      ('criterion',         ('drop', ['gini','entropy'], 'gini', 'Split quality measure')),
    ]))),
])

_SM_LOGIT  = 'Logit (statsmodels)'
_SM_BLURBS = {
  _SM_LOGIT: (
    '<div style="font-size:12px;padding:4px 0">'
    '<a href="https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.Logit.html"'
    ' target="_blank">statsmodels Logit documentation</a> -- '
    'Binary logistic regression with full statistical output: coefficients, standard errors, '
    'p-values, confidence intervals, pseudo-R2, AIC, BIC. Requires a binary target.</div>'),
}

model_dd = widgets.Dropdown(
    options=list(_CATALOG.keys()) + [_SM_LOGIT],
    description='Model:', style={'description_width': '80px'},
    layout=widgets.Layout(width='380px'))
model_dd.observe(_on_model_change, names='value')
_on_model_change(None)

printmd('**Step 2 -- Select model and configure hyperparameters:**')
display(widgets.VBox([model_dd, _model_info, _params_box]))


In [ ]:
# -- Step 3: train and evaluate --------------------------------------------
y_col_raw  = _lbl2raw[y_dd.value]
X_col_raw  = [_lbl2raw[l] for l in X_ms.value if l != y_dd.value]
model_name = model_dd.value
test_size  = split_s.value
rand_seed  = seed_t.value

if not X_col_raw:
    raise ValueError('Select at least one feature column in the X multi-select.')

df_model = df_raw[X_col_raw + [y_col_raw]].dropna(subset=[y_col_raw]).copy()
X_num = [c for c in X_col_raw if _isnumcol(c)]
X_cat = [c for c in X_col_raw if not _isnumcol(c)]

ct = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('scl', StandardScaler())]), X_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OrdinalEncoder(handle_unknown='use_encoded_value',
                                             unknown_value=-1))]), X_cat),
], remainder='drop')

_le       = LabelEncoder()
y_all     = _le.fit_transform(df_model[y_col_raw].astype(str))
classes   = _le.classes_
n_classes = len(classes)
is_binary = n_classes == 2
feat_names = [_lbl(c) for c in X_col_raw]

X_tr, X_te, y_tr, y_te = train_test_split(
    df_model[X_col_raw], y_all, test_size=test_size,
    random_state=rand_seed, stratify=y_all if n_classes <= 20 else None)
X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)

if model_name == 'Logit (statsmodels)':
    if not is_binary:
        raise ValueError('statsmodels Logit requires a binary target.')
    _Xtr    = X_tr_t.toarray() if hasattr(X_tr_t, 'toarray') else X_tr_t
    _Xte    = X_te_t.toarray() if hasattr(X_te_t, 'toarray') else X_te_t
    _sm_fit = sm.Logit(y_tr, sm.add_constant(_Xtr)).fit(disp=False)
    y_pred  = (_sm_fit.predict(sm.add_constant(_Xte, has_constant='add')) > 0.5).astype(int)
    y_prob  = None
    _fitted = None
    display(HTML('<pre style="font-size:11px">' + _sm_fit.summary().as_text() + '</pre>'))
else:
    spec    = _CATALOG[model_name]
    _fitted = spec['cls'](**{**spec['fit_kw'], **_hp_vals()})
    _fitted.fit(X_tr_t, y_tr)
    y_pred  = _fitted.predict(X_te_t)
    y_prob  = (_fitted.predict_proba(X_te_t)[:, 1]
               if is_binary and hasattr(_fitted, 'predict_proba') else None)

acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred,
                        average='binary' if is_binary else 'macro', zero_division=0)
rec  = recall_score(y_te, y_pred,
                     average='binary' if is_binary else 'macro', zero_division=0)
f1   = f1_score(y_te, y_pred,
                 average='binary' if is_binary else 'macro', zero_division=0)
auc  = roc_auc_score(y_te, y_prob) if y_prob is not None else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cm = confusion_matrix(y_te, y_pred)
axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(n_classes))
axes[0].set_xticklabels(classes, rotation=30, ha='right', fontsize=9)
axes[0].set_yticks(range(n_classes))
axes[0].set_yticklabels(classes, fontsize=9)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion matrix -- ' + model_name)
for i in range(n_classes):
    for j in range(n_classes):
        axes[0].text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=10,
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')

mnames = ['Accuracy', 'Precision', 'Recall', 'F1']
mvals  = [acc, prec, rec, f1]
if auc is not None:
    mnames.append('ROC AUC'); mvals.append(auc)
bars = axes[1].bar(mnames, mvals,
                   color=['#6366f1','#0ea5e9','#10b981','#f59e0b','#ec4899'][:len(mvals)])
axes[1].set_ylim(0, 1.12); axes[1].set_title('Test-set performance')
for bar, v in zip(bars, mvals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                 '{:.3f}'.format(v), ha='center', va='bottom',
                 fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

top_note = ''
if _fitted is not None and hasattr(_fitted, 'feature_importances_'):
    imp  = _fitted.feature_importances_
    top3 = [feat_names[i] for i in np.argsort(imp)[::-1][:3]]
    top_note = '  Top features by impurity gain: **{}**, **{}**, **{}**.'.format(*top3)

auc_str  = ' | ROC AUC **{:.3f}**'.format(auc) if auc else ''
x_listed = ', '.join('**' + _lbl(c) + '**' for c in X_col_raw[:5])
x_listed += '...' if len(X_col_raw) > 5 else ''
printmd(
    '**' + model_name + ' results**\n\n'
    'Accuracy **{:.1%}** | Precision **{:.1%}** | Recall **{:.1%}**'
    ' | F1 **{:.1%}**{}  \n'
    'Test set: {:,} samples ({:.0%} of {:,} total).{}\n\n'
    'Predicted **{}** from {} feature(s): {}.'.format(
        acc, prec, rec, f1, auc_str,
        len(y_te), test_size, len(df_model), top_note,
        _lbl(y_col_raw), len(X_col_raw), x_listed)
)
if _fitted is not None:
    print(classification_report(y_te, y_pred,
          target_names=[str(c) for c in classes], zero_division=0))


In [ ]:
# -- Step 4: SHAP feature importance ---------------------------------------
if model_name == 'Logit (statsmodels)' or _fitted is None:
    printmd('*SHAP is not applicable to the statsmodels Logit. '
            'Coefficient signs and magnitudes in the summary above indicate direction of influence.*')
else:
    shap_type = _CATALOG[model_name]['shap']
    printmd('**SHAP feature importance ({} explainer):**'.format(shap_type))
    try:
        X_te_d = X_te_t.toarray() if hasattr(X_te_t, 'toarray') else X_te_t
        X_tr_d = X_tr_t.toarray() if hasattr(X_tr_t, 'toarray') else X_tr_t
        n_te   = min(200, len(X_te_d))
        if shap_type == 'tree':
            expl = shap.TreeExplainer(_fitted)
            sv   = expl.shap_values(X_te_d[:n_te])
        elif shap_type == 'linear':
            bg   = shap.maskers.Independent(X_tr_d, max_samples=100)
            expl = shap.LinearExplainer(_fitted, bg)
            sv   = expl.shap_values(X_te_d[:n_te])
        else:
            bg   = shap.kmeans(X_tr_d[:min(100, len(X_tr_d))], 10)
            expl = shap.KernelExplainer(_fitted.predict_proba, bg)
            sv   = expl.shap_values(X_te_d[:50])

        if isinstance(sv, list):
            mean_abs = np.abs(np.array(sv)).mean(axis=0).mean(axis=0)
        else:
            sv2 = sv if sv.ndim == 2 else sv[:, :, 1]
            mean_abs = np.abs(sv2).mean(axis=0)

        idx = np.argsort(mean_abs)[::-1][:20]
        fig, ax = plt.subplots(figsize=(8, max(4, len(idx) * 0.35 + 1)))
        ax.barh([feat_names[i] for i in idx[::-1]], mean_abs[idx[::-1]], color='#6366f1')
        ax.set_xlabel('Mean |SHAP value| -- average impact on model output')
        ax.set_title('SHAP feature importance -- ' + model_name)
        plt.tight_layout(); plt.show()

        top3 = [feat_names[i] for i in idx[:3]]
        printmd(
            'The three features with greatest average influence: '
            '**{}**, **{}**, **{}**. '
            'SHAP values show how much each feature pushes the model output '
            'away from the base rate, averaged over the test set.'.format(*top3)
        )
    except Exception as e:
        printmd('*SHAP computation failed: {}*'.format(e))


In [ ]:
# -- Step 5a: 5-fold cross-validation -------------------------------------
if model_name == 'Logit (statsmodels)' or _fitted is None:
    printmd('*Automated cross-validation is not available for the statsmodels Logit.*')
else:
    printmd('**5-fold stratified cross-validation:**')
    cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=rand_seed)
    X_full_t = ct.transform(df_model[X_col_raw])

    cv_acc = cross_val_score(_fitted, X_full_t, y_all, cv=cv,
                              scoring='accuracy',  n_jobs=-1)
    cv_f1  = cross_val_score(_fitted, X_full_t, y_all, cv=cv,
                              scoring='f1_macro',  n_jobs=-1)

    x = np.arange(5); w = 0.35
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(x - w/2, cv_acc, w, label='Accuracy',   color='#6366f1', alpha=0.85)
    ax.bar(x + w/2, cv_f1,  w, label='F1 (macro)', color='#0ea5e9', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(['Fold {}'.format(i+1) for i in range(5)])
    ax.axhline(cv_acc.mean(), color='#6366f1', ls='--', lw=1, alpha=0.6)
    ax.axhline(cv_f1.mean(),  color='#0ea5e9', ls='--', lw=1, alpha=0.6)
    ax.set_ylim(0, 1.12); ax.legend(); ax.set_title('5-fold cross-validation scores')
    plt.tight_layout(); plt.show()

    if cv_acc.std() < 0.03:
        var_note = 'consistent across folds -- good generalization'
    elif cv_acc.std() < 0.08:
        var_note = 'moderately variable -- consider more data or stronger regularization'
    else:
        var_note = 'highly variable -- the model may be overfitting or the sample is small'

    printmd(
        'CV Accuracy: **{:.1%} +/- {:.1%}** (5 folds)  \n'
        'CV F1 (macro): **{:.1%} +/- {:.1%}** (5 folds)  \n\n'
        'Performance is **{}** (std = {:.3f}).'.format(
            cv_acc.mean(), cv_acc.std(),
            cv_f1.mean(),  cv_f1.std(),
            var_note, cv_acc.std())
    )


In [ ]:
# -- Step 5b: save predictions and create new survey ----------------------
X_full_t = ct.transform(df_model[X_col_raw])
if model_name == 'Logit (statsmodels)':
    _Xfull    = X_full_t.toarray() if hasattr(X_full_t, 'toarray') else X_full_t
    all_preds = (_sm_fit.predict(
        sm.add_constant(_Xfull, has_constant='add')) > 0.5).astype(int)
else:
    all_preds = _fitted.predict(X_full_t)

pred_col = 'predicted_' + _lbl(y_col_raw)
df_out   = df_raw.copy()
df_out.loc[df_model.index, pred_col] = _le.inverse_transform(all_preds)

new_file    = suaveint.save_csv_file(df_out, absolutePath, csv_file)
survey_name = input('Enter name for new survey: ')
printmd('**Survey name:** ' + survey_name)
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)
